In [1]:
library(spacexr)
library(Seurat)
library(ggplot2)
library(cowplot)
library(dplyr)
library(RColorBrewer)
cols = c(brewer.pal(9, "Set1"),brewer.pal(8,"Set2")[1:8],brewer.pal(12,"Paired")[1:12],brewer.pal(8,"Dark2")[1:8],brewer.pal(8,"Accent"),brewer.pal(12, "Set3"),brewer.pal(9,"Pastel1"),brewer.pal(8,"Pastel2"))
cols = c(cols,cols)

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
profile = 'T993'
sc_data = readRDS("/data/snRNA_downsample.rds") 
st_data = readRDS(paste0("/data/",profile,"cellbin.rds"))

In [4]:
Idents(sc_data) = 'subclass.v4'
counts=sc_data@assays[["RNA"]]@counts
celltype=sc_data@meta.data[["subclass.v4"]]  
celltype2=gsub("/","_",celltype)
celltype2=as.factor(celltype2)
names(celltype2)=rownames(sc_data@meta.data)
counts2=as(counts,"dgCMatrix")
reference=Reference(counts, celltype2, nUMI = NULL, require_int = F, n_max_cells = 10000, min_UMI = 50)

st_count=st_data@assays[["Spatial"]]@counts
st_count <- as(st_count,"matrix")
st_count <- as(st_count,"dgCMatrix")
stereo = st_data@reductions[["stereo"]]@cell.embeddings
colnames(stereo) <- c('x', 'y')
stereo=as.data.frame(stereo)
puck= SpatialRNA(stereo,st_count)

myRCTD <- create.RCTD(puck, reference, max_cores =16, test_mode = FALSE,CELL_MIN_INSTANCE = 3)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 31.7 GiB”
Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 108357

process_cell_type_info: number of genes in reference: 57953




                     Astrocytes                   L2 IT neurons 
                           5000                            4395 
                L2_3 IT neurons                   L3 IT neurons 
                           5645                            4459 
                L3_4 IT neurons                 L3-6 IT neurons 
                            556                            5000 
                  L4 IT neurons                 L4_5 IT neurons 
                           4701                            5244 
                  L5 ET neurons               L5_6 CAR3 neurons 
                            761                            5000 
                L5_6 NP neurons                   L6 CT neurons 
                           4556                            5000 
                  L6 IT neurons                     L6b neurons 
                           5000                            5000 
                  LAMP5 neurons                       Microglia 
                        

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.9 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.4 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.9 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.0 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.0 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(ob

In [ ]:
myRCTD1 <- run.RCTD(myRCTD, doublet_mode = 'doublet')
results_df=myRCTD1@results$results_df  
cell_type_names <- myRCTD1@cell_type_info$info[[2]]
results_df$x <- stereo$x[match(rownames(results_df),rownames(stereo))]
results_df$y <- stereo$y[match(rownames(results_df),rownames(stereo))]
save(myRCTD1,results_df,file=paste0('/data/RCTD_',profile,".cellbin_prct.Rdata"))
write.csv(myRCTD1@results$results_df,paste0('/data/RCTD_',profile,".cellbin_prct.csv"))

fitBulk: decomposing bulk

chooseSigma: using initial Q_mat with sigma =  1

Likelihood value: 746636.312078389

Sigma value:  1

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
